# Session 4 — Student Solution

## Messy Tabular Files: Excel and Schema

This annotated solution shows one defensible reconstruction. The important part is not the exact syntax. It is the evidence used to identify observations, exclude reporting totals and define the expected table.


In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)


def locate_file(filename: str) -> Path:
    """Search common classroom folders and return the first matching file."""
    candidates = [
        Path.cwd() / filename,
        Path.cwd() / "data" / filename,
        Path.cwd() / "Students" / filename,
        Path("/mnt/data") / filename,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    searched = "\n".join(f"- {path}" for path in candidates)
    raise FileNotFoundError(f"Could not find {filename}. Searched:\n{searched}")


DATA_FILE = locate_file("S4_student_regional_sales_challenge.xlsx")
print(f"Using: {DATA_FILE.name}")


Using: S4_student_regional_sales_challenge.xlsx


## 1. Inspect the workbook before selecting sheets


In [2]:
# The workbook structure is evidence. We do not choose a sheet by position.
excel_file = pd.ExcelFile(DATA_FILE)
excel_file.sheet_names


['README', 'Executive Summary', 'ES Report', 'FR Report', 'DE Report']

In [3]:
# Inspect all cells from one country report to locate titles, headers and totals.
raw_es = pd.read_excel(DATA_FILE, sheet_name="ES Report", header=None)
raw_es.iloc[:20, :6]


,0,1,2,3,4,5
0,Spain commercial report,NaN,NaN,NaN,NaN,NaN
1,Regional Sales Report — Q2 2026,NaN,NaN,NaN,NaN,NaN
2,Prepared for,Commercial Steering Committee,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN
4,Market,Period,Channel,Activity,NaN,Net sales (EUR)
5,NaN,NaN,NaN,Orders,Units,NaN
6,Spain,Apr 2026,Online,420,615,"€ 68 250,00"
7,Spain,Apr 2026,Store,310,455,"€ 51 200,00"
8,SUBTOTAL,Apr 2026,NaN,730,1070,"€ 119 450,00"
9,Spain,May 2026,Online,460,690,"€ 78 200,00"


The `README` describes the business scope. `Executive Summary` is a valid management view, but its grain is one row per country. The analytical observations are in the three country report sheets.

Declared grain:

> **One row = one country × one month × one channel**

The channel field is populated only for detail observations, so it provides reproducible evidence for excluding subtotals, the grand total and notes.


## 2. Convert the monetary representations


In [4]:
def parse_euro(value):
    """Convert numeric, Spanish-style and German-style EUR values to float."""
    if pd.isna(value):
        return pd.NA

    # France stores the amount as a real Excel number.
    if isinstance(value, (int, float)):
        return float(value)

    # Spain uses '€ 68 250,00'; Germany uses '68.900,00 €'.
    text = str(value).replace("€", "").replace("\u00a0", " ").strip()
    text = text.replace(" ", "").replace(".", "").replace(",", ".")
    return float(text)


examples = ["€ 68 250,00", 67500, "62.400,00 €"]
[parse_euro(value) for value in examples]


[68250.0, 67500.0, 62400.0]

## 3. Extract only the observations


In [5]:
CANONICAL_COLUMNS = [
    "country",
    "month",
    "channel",
    "orders",
    "units",
    "net_sales_eur",
]
DETAIL_CHANNELS = {"Online", "Store"}


def extract_report_sheet(path: Path, sheet_name: str) -> pd.DataFrame:
    """Reconstruct the analytical rows contained in one country report."""
    raw = pd.read_excel(path, sheet_name=sheet_name, header=None)

    # Excel rows 1–6 are report titles and a two-level presentation header.
    block = raw.iloc[6:, :6].copy()
    block.columns = CANONICAL_COLUMNS

    # The declared grain requires a channel. Totals and notes have no detail channel.
    detail = block[block["channel"].isin(DETAIL_CHANNELS)].copy()

    # Convert source representations into the canonical analytical types.
    detail["month"] = pd.to_datetime(detail["month"], format="%b %Y")
    detail["orders"] = pd.to_numeric(detail["orders"], errors="raise").astype("int64")
    detail["units"] = pd.to_numeric(detail["units"], errors="raise").astype("int64")
    detail["net_sales_eur"] = detail["net_sales_eur"].map(parse_euro).astype("float64")

    return detail.reset_index(drop=True)


REPORT_SHEETS = ["ES Report", "FR Report", "DE Report"]
frames = [extract_report_sheet(DATA_FILE, sheet) for sheet in REPORT_SHEETS]
canonical = pd.concat(frames, ignore_index=True)

print("Canonical shape:", canonical.shape)
canonical.head(8)


Canonical shape: (18, 6)


,country,month,channel,orders,units,net_sales_eur
0,Spain,2026-04-01,Online,420,615,"68,250.00"
1,Spain,2026-04-01,Store,310,455,"51,200.00"
2,Spain,2026-05-01,Online,460,690,"78,200.00"
3,Spain,2026-05-01,Store,325,480,"54,800.00"
4,Spain,2026-06-01,Online,500,755,"86,600.00"
5,Spain,2026-06-01,Store,340,510,"58,900.00"
6,France,2026-04-01,Online,390,580,"67,500.00"
7,France,2026-04-01,Store,360,530,"64,800.00"


## 4. Measure the effect of keeping report totals


In [6]:
def sales_sum_including_report_totals(path: Path, sheet_name: str) -> float:
    """Calculate the misleading total produced when reporting aggregates remain."""
    raw = pd.read_excel(path, sheet_name=sheet_name, header=None)
    block = raw.iloc[6:, :6].copy()
    block.columns = CANONICAL_COLUMNS
    return block["net_sales_eur"].dropna().map(parse_euro).sum()


wrong_total = sum(sales_sum_including_report_totals(DATA_FILE, sheet) for sheet in REPORT_SHEETS)
correct_total = canonical["net_sales_eur"].sum()

print(f"With subtotals and grand totals: €{wrong_total:,.0f}")
print(f"Detail observations only:       €{correct_total:,.0f}")
print(f"Overstatement factor:           {wrong_total / correct_total:.1f}×")


With subtotals and grand totals: €3,828,150
Detail observations only:       €1,276,050
Overstatement factor:           3.0×


## 5. Validate scope, grain and values


In [7]:
expected_countries = {"Spain", "France", "Germany"}
expected_months = pd.to_datetime(["2026-04-01", "2026-05-01", "2026-06-01"])
expected_channels = {"Online", "Store"}
expected_rows = len(expected_countries) * len(expected_months) * len(expected_channels)

checks = {
    "exact canonical columns": list(canonical.columns) == CANONICAL_COLUMNS,
    "expected row count": len(canonical) == expected_rows,
    "expected countries": set(canonical["country"]) == expected_countries,
    "expected months": set(canonical["month"]) == set(expected_months),
    "expected channels": set(canonical["channel"]) == expected_channels,
    "unique business key": not canonical.duplicated(["country", "month", "channel"]).any(),
    "required values present": not canonical[CANONICAL_COLUMNS].isna().any().any(),
    "non-negative measures": canonical[["orders", "units", "net_sales_eur"]].ge(0).all().all(),
}

check_results = pd.Series(checks, name="passed")
assert check_results.all(), check_results[~check_results]
check_results


exact canonical columns    True
expected row count         True
expected countries         True
expected months            True
expected channels          True
unique business key        True
required values present    True
non-negative measures      True
Name: passed, dtype: bool

## 6. Validate the explicit schema

Pandera turns the expected columns, types, nullability, allowed categories, numeric rules and business key into one reusable contract.


In [ ]:
import pandera.pandas as pa

sales_schema = pa.DataFrameSchema(
    {
        "country": pa.Column(
            str,
            pa.Check.isin(["Spain", "France", "Germany"]),
            nullable=False,
        ),
        "month": pa.Column("datetime64[ns]", nullable=False),
        "channel": pa.Column(
            str,
            pa.Check.isin(["Online", "Store"]),
            nullable=False,
        ),
        "orders": pa.Column(int, pa.Check.ge(0), nullable=False),
        "units": pa.Column(int, pa.Check.ge(0), nullable=False),
        "net_sales_eur": pa.Column(float, pa.Check.ge(0), nullable=False),
    },
    strict=True,
    ordered=True,
    coerce=True,
    unique=["country", "month", "channel"],
)

validated = sales_schema.validate(canonical, lazy=True)
print(f"Schema validation passed for {len(validated)} rows.")


## 7. Answer the business question


In [8]:
market_channel_sales = (
    canonical.groupby(["country", "channel"], as_index=False)["net_sales_eur"]
    .sum()
    .sort_values("net_sales_eur", ascending=False)
    .reset_index(drop=True)
)
market_channel_sales


,country,channel,net_sales_eur
0,Germany,Store,"245,900.00"
1,Spain,Online,"233,050.00"
2,France,Online,"220,000.00"
3,France,Store,"206,100.00"
4,Germany,Online,"206,100.00"
5,Spain,Store,"164,900.00"


In [9]:
winner = market_channel_sales.iloc[0]
print(
    f"Germany — Store generated the highest validated Q2 net sales: "
    f"€{winner['net_sales_eur']:,.0f}."
)


Germany — Store generated the highest validated Q2 net sales: €245,900.


## 8. Source Passport

- **Source and owner:** regional commercial reporting pack supplied by the regional teams
- **Access method:** internal Excel workbook
- **Authorisation and reuse:** supplied for internal analysis
- **Reporting period:** Q2 2026
- **Expected scope:** Spain, France and Germany; April–June; Online and Store
- **Raw-content location:** immutable workbook used by this notebook
- **Known limitation:** future reports may change their sheet names, header positions or formatting

## 9. Example AI Audit

- **Proposal:** AI suggested loading every sheet and concatenating the results.
- **Verification:** the README and visual structure showed that `Executive Summary` had a different grain.
- **Decision:** reject that sheet and combine only the three country detail reports.
- **Residual limitation:** the extraction still assumes that the source layout begins at the same row in future files.

## 10. Defensible Claim

- **Claim:** Germany Store generated the highest validated Q2 net sales, at €245,900.
- **Evidence:** 18 detail observations at the declared grain, complete expected scope, unique business key and passing schema.
- **Assumption:** rows without a valid detail channel are reporting aggregates or notes, not observations.
- **Limitation:** schema validation cannot detect a plausible but incorrect upstream sales amount.
- **Confidence:** high for the supplied workbook and stated scope.


## Lessons learned

1. Workbook structure must be interpreted before data can be combined.
2. The declared grain provides evidence for excluding report totals.
3. A canonical representation separates source formatting from analytical meaning.
4. Scope checks and key checks complement the schema.
5. A schema tests conformity with expectations, not business truth.
